In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from torch.utils.data import DataLoader, TensorDataset
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df_cleaned = pd.read_csv("df_cleaned.csv")

In [21]:
from pathlib import Path
from imblearn.over_sampling import SMOTE
import joblib

# 1) Remove the bottom 4 rare classes first by keeping only the planned 11 classes
target_counts = {
    0: 8000,
    4: 8000,
    2: 8000,
    3: 8000,
    7: 7000,
    6: 7000,
    5: 7000,
    11: 6500,
    10: 6000,
    12: 5500,
    1: 5500,
}

keep_labels = sorted(target_counts.keys())
df_model = df_cleaned[df_cleaned["Label"].isin(keep_labels)].copy()

# Build contiguous label ids for model training (0..10)
original_to_current = {orig_label: idx for idx, orig_label in enumerate(keep_labels)}
current_to_original = {idx: orig_label for orig_label, idx in original_to_current.items()}

# 2) Train-test split on filtered dataframe
train_df_raw, test_df_raw = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42,
    stratify=df_model["Label"]
)

feature_cols = [c for c in train_df_raw.columns if c != "Label"]

# 3) Fit scaler on TRAIN split only, then transform TRAIN and TEST with same stats
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df_raw[feature_cols])
X_test_scaled = scaler.transform(test_df_raw[feature_cols])

train_df_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols)
train_df_scaled["Label"] = train_df_raw["Label"].to_numpy()

test_df = pd.DataFrame(X_test_scaled, columns=feature_cols)
test_df["Label"] = test_df_raw["Label"].to_numpy()

scaler_path = Path("feature_scaler.joblib")
joblib.dump(scaler, scaler_path)

# 4) Balance training split only: undersample large classes, then SMOTE oversample small classes
under_parts = []
for cls, target_n in target_counts.items():
    cls_rows = train_df_scaled[train_df_scaled["Label"] == cls]
    if len(cls_rows) > target_n:
        cls_rows = cls_rows.sample(n=target_n, random_state=42)
    under_parts.append(cls_rows)

train_df_under = (
    pd.concat(under_parts, axis=0)
    .sample(frac=1.0, random_state=42)
    .reset_index(drop=True)
)

X_under = train_df_under[feature_cols]
y_under = train_df_under["Label"]

current_counts = y_under.value_counts().to_dict()
smote_strategy = {
    cls: target_n
    for cls, target_n in target_counts.items()
    if current_counts.get(cls, 0) < target_n
}

if smote_strategy:
    smote = SMOTE(
        sampling_strategy=smote_strategy,
        random_state=42,
        k_neighbors=5
    )
    X_balanced, y_balanced = smote.fit_resample(X_under, y_under)
    train_df = pd.DataFrame(X_balanced, columns=feature_cols)
    train_df["Label"] = y_balanced
else:
    train_df = train_df_under.copy()

# Convert original labels to contiguous labels used by the model
train_df["Label"] = train_df["Label"].map(original_to_current).astype(int)
test_df["Label"] = test_df["Label"].map(original_to_current).astype(int)

train_df = train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Save prepared datasets for model training/evaluation
train_data_path = Path("train_balanced_smote.csv")
test_data_path = Path("test_filtered_current_labels.csv")
train_df.to_csv(train_data_path, index=False)
test_df.to_csv(test_data_path, index=False)

# 5) Keep persistent label lineage info so later remapping is never lost
label_map_path = Path("label_mapping_tracker.csv")
base_label_map = pd.DataFrame({
    "original_label": keep_labels,
    "current_label": [original_to_current[lbl] for lbl in keep_labels],
})

if label_map_path.exists():
    existing_map = pd.read_csv(label_map_path)
    for col in ["final_label", "final_attack_name"]:
        if col not in existing_map.columns:
            existing_map[col] = pd.NA
    label_mapping_tracker = base_label_map.merge(
        existing_map[["original_label", "final_label", "final_attack_name"]],
        on="original_label",
        how="left"
    )
else:
    label_mapping_tracker = base_label_map.assign(
        final_label=pd.NA,
        final_attack_name=pd.NA
    )

label_mapping_tracker = label_mapping_tracker.sort_values("current_label").reset_index(drop=True)
label_mapping_tracker.to_csv(label_map_path, index=False)

# Optional but useful: save per-label counts at each stage (indexed by original labels)
train_before_counts = train_df_raw["Label"].value_counts().reindex(keep_labels).fillna(0).astype(int)
train_after_counts = train_df["Label"].map(current_to_original).value_counts().reindex(keep_labels).fillna(0).astype(int)
test_counts = test_df["Label"].map(current_to_original).value_counts().reindex(keep_labels).fillna(0).astype(int)

label_count_tracker = pd.DataFrame({
    "original_label": keep_labels,
    "current_label": [original_to_current[lbl] for lbl in keep_labels],
    "filtered_total": df_model["Label"].value_counts().reindex(keep_labels).fillna(0).astype(int).values,
    "train_before_balance": train_before_counts.values,
    "train_after_balance": train_after_counts.values,
    "test_total": test_counts.values,
})
label_count_tracker.to_csv("label_count_tracker.csv", index=False)

print("Filtered original classes kept:", keep_labels)
print("\nOriginal -> current label map:")
print(label_mapping_tracker[["original_label", "current_label"]])
print("\nSMOTE strategy used (original labels):", smote_strategy if smote_strategy else "No SMOTE needed")
print("\nBalanced train class counts (current labels):")
print(train_df["Label"].value_counts().sort_index())
print("\nTest class counts (current labels):")
print(test_df["Label"].value_counts().sort_index())
print(f"\nBalanced train size: {len(train_df):,}")
print(f"Test size: {len(test_df):,}")
print(f"\nSaved train dataset: {train_data_path}")
print(f"Saved test dataset: {test_data_path}")
print(f"Saved fitted scaler: {scaler_path}")
print("Saved label lineage template: label_mapping_tracker.csv")
print("Saved stage-wise counts: label_count_tracker.csv")

Filtered original classes kept: [0, 1, 2, 3, 4, 5, 6, 7, 10, 11, 12]

Original -> current label map:
    original_label  current_label
0                0              0
1                1              1
2                2              2
3                3              3
4                4              4
5                5              5
6                6              6
7                7              7
8               10              8
9               11              9
10              12             10

SMOTE strategy used (original labels): {7: 7000, 6: 7000, 5: 7000, 11: 6500, 10: 6000, 12: 5500, 1: 5500}

Balanced train class counts (current labels):
Label
0     8000
1     5500
2     8000
3     8000
4     8000
5     7000
6     7000
7     7000
8     6000
9     6500
10    5500
Name: count, dtype: int64

Test class counts (current labels):
Label
0     379334
1        287
2      25603
3       2057
4      34569
5       1046
6       1077
7       1186
8        391
9        644
10       29

In [4]:
CONFIG = {
    # data
    "train_data_path": "train_balanced_smote.csv",
    "test_data_path": "test_filtered_current_labels.csv",
    "num_classes": 11,

    # transformer architecture
    "d_model": 64,
    "n_heads": 4,
    "depth": 2,
    "dropout": 0.1,

    # training
    "batch_size": 256,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "epochs": 20,

    # experiment name
    "experiment_name": "baseline_v1"
}

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [14]:
class FeatureTokenizer(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.value_projection = nn.Linear(1, d_model)
        self.feature_embedding = nn.Parameter(
            torch.randn(num_features, d_model)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.value_projection(x)
        x = x + self.feature_embedding
        return x


class TabularTransformer(nn.Module):
    def __init__(self, num_features, num_classes,
                 d_model, n_heads, depth, dropout):
        super().__init__()

        self.tokenizer = FeatureTokenizer(num_features, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        batch_size = x.size(0)

        x = self.tokenizer(x)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)

        cls_output = x[:, 0]
        logits = self.classifier(cls_output)

        return logits

In [22]:
def get_dataloaders(config):
    from pathlib import Path

    train_path = Path(config["train_data_path"])
    test_path = Path(config["test_data_path"])

    if not train_path.exists() or not test_path.exists():
        raise RuntimeError(
            "Run Cell 3 first so prepared train/test CSV files are created."
        )

    train_data = pd.read_csv(train_path)
    test_data = pd.read_csv(test_path)

    feature_cols = [c for c in train_data.columns if c != "Label"]
    missing_cols = [c for c in feature_cols if c not in test_data.columns]
    if missing_cols:
        raise RuntimeError(
            f"Test dataset is missing feature columns: {missing_cols[:5]}"
        )

    # Features are already standardized in preprocessing (fit on train only).
    X_train = train_data[feature_cols].astype(np.float32).values
    y_train = train_data["Label"].astype(int).values

    X_test = test_data[feature_cols].astype(np.float32).values
    y_test = test_data["Label"].astype(int).values

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)

    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=config["batch_size"],
        shuffle=True
    )

    test_loader = DataLoader(
        TensorDataset(X_test, y_test),
        batch_size=config["batch_size"]
    )

    return train_loader, test_loader, X_train.shape[1]

In [16]:
def train_model(model, train_loader, config):

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    for epoch in range(config["epochs"]):

        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for xb, yb in train_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            outputs = model(xb)
            loss = criterion(outputs, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct += (outputs.argmax(1) == yb).sum().item()
            total += yb.size(0)

        print(
            f"Epoch {epoch+1} | "
            f"Loss: {total_loss/len(train_loader):.4f} | "
            f"Train Acc: {correct/total:.4f}"
        )

In [17]:
import torch
import pandas as pd
from sklearn.metrics import classification_report

def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    # 1. Run the evaluation loop first
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            outputs = model(xb)
            preds = outputs.argmax(1)

            correct += (preds == yb).sum().item()
            total += yb.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    # 2. Calculate simple accuracy
    accuracy = correct / total

    # 3. Use current label mapping from Cell 3 lineage tracker
    mapping_df = pd.read_csv("label_mapping_tracker.csv")
    mapping_df["display_name"] = mapping_df["final_attack_name"].where(
        mapping_df["final_attack_name"].notna(),
        "orig_" + mapping_df["original_label"].astype(str)
    )
    label_dict = dict(zip(
        mapping_df["current_label"].astype(int),
        mapping_df["display_name"]
    ))

    true_names = [label_dict.get(int(t), f"class_{t}") for t in all_labels]
    pred_names = [label_dict.get(int(p), f"class_{p}") for p in all_preds]

    # 4. Generate and print reports
    print("Test Accuracy:", accuracy)
    print(classification_report(true_names, pred_names))

    return accuracy

In [18]:
EXPERIMENT_RESULTS = {}

def run_experiment(config):

    print("\n" + "="*60)
    print("Running Experiment:", config["experiment_name"])
    print("="*60)

    # Print major hyperparameters
    print("Hyperparameters:")
    print(f"d_model       : {config['d_model']}")
    print(f"depth         : {config['depth']}")
    print(f"dropout       : {config['dropout']}")
    print(f"lr            : {config['lr']}")
    print(f"batch_size    : {config['batch_size']}")
    print(f"epochs        : {config['epochs']}")
    print("-"*60)

    # ---------------- DATA ----------------
    train_loader, test_loader, num_features = get_dataloaders(config)

    model = TabularTransformer(
        num_features=num_features,
        num_classes=config["num_classes"],
        d_model=config["d_model"],
        n_heads=config["n_heads"],
        depth=config["depth"],
        dropout=config["dropout"]
    ).to(device)

    # ---------------- TRAIN ----------------
    train_model(model, train_loader, config)

    # ---------------- EVALUATE ----------------
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            outputs = model(xb)
            preds = outputs.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    # Map current labels to readable names using lineage tracker
    mapping_df = pd.read_csv("label_mapping_tracker.csv")
    mapping_df["display_name"] = mapping_df["final_attack_name"].where(
        mapping_df["final_attack_name"].notna(),
        "orig_" + mapping_df["original_label"].astype(str)
    )
    label_dict = dict(zip(
        mapping_df["current_label"].astype(int),
        mapping_df["display_name"]
    ))

    true_names = [label_dict.get(int(t), f"class_{t}") for t in all_labels]
    pred_names = [label_dict.get(int(p), f"class_{p}") for p in all_preds]

    report = classification_report(true_names, pred_names)

    accuracy = np.mean(np.array(true_names) == np.array(pred_names))

    print(f"\nTest Accuracy: {accuracy:.4f}")
    print("\nClassification Report:\n")
    print(report)

    # ---------------- STORE RESULTS ----------------
    EXPERIMENT_RESULTS[config["experiment_name"]] = {
        "hyperparameters": {
            "d_model": config["d_model"],
            "depth": config["depth"],
            "dropout": config["dropout"],
            "lr": config["lr"],
            "batch_size": config["batch_size"],
            "epochs": config["epochs"]
        },
        "accuracy": accuracy,
        "report": report
    }

    print("Experiment stored.\n")

    return accuracy

In [24]:
run_experiment(CONFIG)


Running Experiment: high_precision
Hyperparameters:
d_model       : 64
depth         : 2
dropout       : 0.3
lr            : 0.0001
batch_size    : 256
epochs        : 20
------------------------------------------------------------
Epoch 1 | Loss: 2.0226 | Train Acc: 0.3156
Epoch 2 | Loss: 1.5003 | Train Acc: 0.5966
Epoch 3 | Loss: 1.1979 | Train Acc: 0.7539
Epoch 4 | Loss: 1.0610 | Train Acc: 0.8032
Epoch 5 | Loss: 1.0025 | Train Acc: 0.8209
Epoch 6 | Loss: 0.9562 | Train Acc: 0.8365
Epoch 7 | Loss: 0.9101 | Train Acc: 0.8554
Epoch 8 | Loss: 0.8761 | Train Acc: 0.8660
Epoch 9 | Loss: 0.8480 | Train Acc: 0.8760
Epoch 10 | Loss: 0.8212 | Train Acc: 0.8872
Epoch 11 | Loss: 0.8024 | Train Acc: 0.8945
Epoch 12 | Loss: 0.7841 | Train Acc: 0.9002
Epoch 13 | Loss: 0.7687 | Train Acc: 0.9053
Epoch 14 | Loss: 0.7551 | Train Acc: 0.9109
Epoch 15 | Loss: 0.7430 | Train Acc: 0.9147
Epoch 16 | Loss: 0.7336 | Train Acc: 0.9196
Epoch 17 | Loss: 0.7229 | Train Acc: 0.9225
Epoch 18 | Loss: 0.7128 | Tr

0.8629974377810826

In [25]:
experiments = [
    {"name":"baseline",       "d_model":64,  "depth":2, "dropout":0.1},
    {"name":"bigger_model",   "d_model":128, "depth":4, "dropout":0.1},
    {"name":"high_precision", "d_model":64,  "depth":2, "dropout":0.3},
    {"name":"high_recall",    "d_model":128, "depth":4, "dropout":0.05},
    {"name":"small_model",    "d_model":32,  "depth":1, "dropout":0.2},
]
""
for exp in experiments:

    CONFIG["experiment_name"] = exp["name"]
    CONFIG["d_model"] = exp["d_model"]
    CONFIG["depth"] = exp["depth"]
    CONFIG["dropout"] = exp["dropout"]

    run_experiment(CONFIG)


Running Experiment: baseline
Hyperparameters:
d_model       : 64
depth         : 2
dropout       : 0.1
lr            : 0.0001
batch_size    : 256
epochs        : 20
------------------------------------------------------------
Epoch 1 | Loss: 1.9185 | Train Acc: 0.3717
Epoch 2 | Loss: 1.1545 | Train Acc: 0.7645
Epoch 3 | Loss: 0.9035 | Train Acc: 0.8598
Epoch 4 | Loss: 0.8111 | Train Acc: 0.8921
Epoch 5 | Loss: 0.7620 | Train Acc: 0.9085
Epoch 6 | Loss: 0.7274 | Train Acc: 0.9220
Epoch 7 | Loss: 0.7020 | Train Acc: 0.9321
Epoch 8 | Loss: 0.6839 | Train Acc: 0.9397
Epoch 9 | Loss: 0.6651 | Train Acc: 0.9481
Epoch 10 | Loss: 0.6552 | Train Acc: 0.9520
Epoch 11 | Loss: 0.6449 | Train Acc: 0.9542
Epoch 12 | Loss: 0.6382 | Train Acc: 0.9566
Epoch 13 | Loss: 0.6333 | Train Acc: 0.9588
Epoch 14 | Loss: 0.6285 | Train Acc: 0.9593
Epoch 15 | Loss: 0.6260 | Train Acc: 0.9607
Epoch 16 | Loss: 0.6218 | Train Acc: 0.9618
Epoch 17 | Loss: 0.6186 | Train Acc: 0.9628
Epoch 18 | Loss: 0.6159 | Train Ac

In [23]:
train_loader, test_loader, num_features = get_dataloaders(CONFIG)

xb, yb = next(iter(train_loader))
xt, yt = next(iter(test_loader))

print("Train batch mean/std:", float(xb.mean()), float(xb.std()))
print("Test batch mean/std:", float(xt.mean()), float(xt.std()))
print("Num features:", num_features)
print("Train batch shape:", tuple(xb.shape), "| Test batch shape:", tuple(xt.shape))

Train batch mean/std: 0.15112870931625366 1.3151508569717407
Test batch mean/std: -0.014892427250742912 0.8377161026000977
Num features: 58
Train batch shape: (256, 58) | Test batch shape: (256, 58)
